<a href="https://colab.research.google.com/github/shyampandey263/gobardhan-guidelines-chatbot/blob/main/GOBARdhan_Guidelines_Chatbot_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# ---------- Install OCR tools ----------
!apt-get update -q
!apt-get install -y -q tesseract-ocr poppler-utils
!pip install -q pytesseract pdf2image pypdf

# ---------- Create the Drive folder and copy the PDF into it ----------
import os, glob, shutil
from pdf2image import convert_from_path
from pypdf import PdfReader
import pytesseract

folder = "/content/drive/MyDrive/gobardhan_bot"
parts  = f"{folder}/ocr_parts"
os.makedirs(parts, exist_ok=True)

found = glob.glob("/content/*.pdf")
assert found, "No PDF found in /content. Upload it with the folder icon first."
pdf_path = f"{folder}/guidelines.pdf"
shutil.copy(found[0], pdf_path)
print("Copied:", found[0], "->", pdf_path)

# ---------- OCR with progress saved to Drive ----------
total_pages = len(PdfReader(pdf_path).pages)
print("Total pages:", total_pages)

for start in range(1, total_pages + 1, 10):
    end = min(start + 9, total_pages)
    part_file = f"{parts}/pages_{start:03d}_{end:03d}.txt"
    if os.path.exists(part_file):
        print(f"Pages {start}-{end} already done, skipping")
        continue
    batch = convert_from_path(pdf_path, dpi=200, first_page=start, last_page=end)
    batch_text = "\n".join(pytesseract.image_to_string(img) for img in batch)
    with open(part_file, "w") as f:
        f.write(batch_text)
    print(f"Pages {start}-{end} done and saved")

# ---------- Merge all parts into one text file ----------
text = ""
for p in sorted(glob.glob(f"{parts}/pages_*.txt")):
    text += open(p).read() + "\n"
text = text.replace("¥", "₹")      # OCR confused the rupee and yen symbols

with open(f"{folder}/gobardhan_text.txt", "w") as f:
    f.write(text)

print(len(text), "characters saved")
print("Files in folder:", os.listdir(folder))

In [ ]:
# ---------- 1. Install libraries ----------
!pip install -q transformers accelerate gradio sentence-transformers faiss-cpu rank_bm25

# ---------- 2. Load the saved text from Drive ----------
import os, re

TEXT_PATH = "/content/drive/MyDrive/gobardhan_bot/gobardhan_text.txt"
assert os.path.exists(TEXT_PATH), "Saved text not found in Drive."
text = open(TEXT_PATH).read().replace("¥", "₹")
print(len(text), "characters loaded")

# ---------- 3. Load the language model ----------
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id, torch_dtype=torch.float16, device_map="auto"
)

def generate_reply(messages, max_new_tokens=300):
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False
        )
    new_tokens = output[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

# ---------- 4. Split the text into chunks ----------
def split_long(p, max_chars):
    if len(p) <= max_chars:
        return [p]
    sentences = re.split(r"(?<=[.;:])\s+", p)
    parts, cur = [], ""
    for s in sentences:
        if len(cur) + len(s) + 1 <= max_chars:
            cur = (cur + " " + s).strip()
        else:
            if cur:
                parts.append(cur)
            while len(s) > max_chars:
                cut = s.rfind(" ", 0, max_chars)
                cut = cut if cut > 0 else max_chars
                parts.append(s[:cut])
                s = s[cut:].strip()
            cur = s
    if cur:
        parts.append(cur)
    return parts

def chunk_paragraphs(text, max_chars=700):
    text = re.sub(r"Page \d+ of \d+", " ", text)
    paras = [re.sub(r"\s+", " ", p).strip() for p in re.split(r"\n\s*\n", text)]
    paras = [p for p in paras if len(p) > 3]
    pieces = [part for p in paras for part in split_long(p, max_chars)]
    chunks, cur = [], ""
    for piece in pieces:
        if len(cur) + len(piece) + 1 <= max_chars:
            cur = (cur + " " + piece).strip()
        else:
            if cur:
                chunks.append(cur)
            cur = piece
    if cur:
        chunks.append(cur)
    return chunks

docs = chunk_paragraphs(text)

# ---------- 5. Build the search (semantic + keyword) ----------
import numpy as np, faiss
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

embedder = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedder.encode(docs, normalize_embeddings=True, show_progress_bar=True)
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(np.array(embeddings, dtype="float32"))

def tokenize(s):
    return re.findall(r"\w+", s.lower())

bm25 = BM25Okapi([tokenize(d) for d in docs])

def retrieve(query, k=4):
    q = embedder.encode([query], normalize_embeddings=True)
    _, sem_ids = index.search(np.array(q, dtype="float32"), 20)
    sem_ids = [int(i) for i in sem_ids[0]]
    kw_scores = bm25.get_scores(tokenize(query))
    kw_ids = [int(i) for i in np.argsort(kw_scores)[::-1][:20]]
    fused = {}
    for ranking in (sem_ids, kw_ids):
        for rank, i in enumerate(ranking):
            fused[i] = fused.get(i, 0) + 1 / (60 + rank)
    top = sorted(fused, key=fused.get, reverse=True)[:k]
    return [docs[i] for i in top]

# Page 1 of the document: the real definition of the scheme
overview = re.sub(r"\s+", " ", text[:1500]).strip()

# ---------- 6. Statistics ----------
n_chunks  = len(docs)
n_vectors = index.ntotal
dim       = embeddings.shape[1]
size_mb   = n_vectors * dim * 4 / (1024 * 1024)

emb_counts = [len(embedder.tokenizer.encode(d, add_special_tokens=False)) for d in docs]
llm_counts = [len(tokenizer.encode(d)) for d in docs]
total_text_tokens = len(tokenizer.encode(text))
limit = embedder.max_seq_length
too_long = sum(c > limit for c in emb_counts)

sample = retrieve("What is the GOBARdhan scheme?", k=5)
prompt_tokens = sum(len(tokenizer.encode(c)) for c in sample) + len(tokenizer.encode(overview))

print("\n=== CHUNKS AND VECTORS ===")
print("Characters in document      :", len(text))
print("Chunks created              :", n_chunks)
print("Vectors in FAISS index      :", n_vectors)
print("Numbers per vector          :", dim)
print(f"Memory used by vectors      : {size_mb:.2f} MB")
print("\n=== TOKENS (chat model tokenizer) ===")
print("Whole document              :", total_text_tokens)
print("Total across all chunks     :", sum(llm_counts))
print("Average per chunk           :", sum(llm_counts) // n_chunks)
print("Longest chunk               :", max(llm_counts))
print("\n=== TOKENS (search model tokenizer) ===")
print("Search model reads up to    :", limit, "tokens per chunk")
print("Chunks longer than that     :", too_long, "(extra text is ignored)")
print("\n=== ONE QUESTION ===")
print("Approx. tokens of context sent to the chat model:", prompt_tokens)

# ---------- 7. Helper used by the chat cell ----------
def to_text(content):
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return " ".join(p.get("text", "") for p in content if isinstance(p, dict))
    return str(content)

In [ ]:
import gradio as gr, re
gr.close_all()

def is_broad(q):
    q = q.lower().strip()
    return len(q.split()) <= 10 and any(
        q.startswith(w) for w in ("what is", "what's", "explain", "describe", "tell me about", "give an overview")
    ) and ("scheme" in q or "gobardhan" in q) and "component" not in q

BROAD_FORMAT = (
    "Answer in exactly this format, using only the DOCUMENT OVERVIEW:\n"
    "1. One sentence: what the scheme is and who approved it.\n"
    "2. Total outlay and the period it covers.\n"
    "3. A numbered list of the six components.\n"
    "Do not add anything else."
)

def llm_chat(message, history):
    question = to_text(message)
    broad = is_broad(question)

    excerpts = [] if broad else retrieve(question, k=5)
    context = "\n\n".join(excerpts) if excerpts else "(none)"

    system = (
        "You answer questions about the GOBARdhan scheme guidelines.\n"
        "RULES:\n"
        "1. Use ONLY the text below. Copy numbers, names and dates exactly as written.\n"
        "2. Do NOT calculate, estimate or infer anything.\n"
        "3. Do NOT add any sentence about aims, benefits or purpose unless the text states it.\n"
        "4. If a date or detail is not in the text, do not mention it.\n"
        "5. The DOCUMENT OVERVIEW is the summary of the whole scheme. The EXCERPTS are "
        "details from specific sections. Do not attribute a detail from a section to the whole scheme.\n"
        "6. If the answer is not in the text, say: 'I could not find this in the document.'\n"
        "7. Keep the answer short and stop after answering.\n\n"
        f"DOCUMENT OVERVIEW:\n{overview}\n\n"
        f"EXCERPTS:\n{context}"
    )
    if broad:
        system += "\n\n" + BROAD_FORMAT

    clean_history = [
        {"role": h["role"], "content": to_text(h["content"])} for h in history[-4:]
    ]
    messages = [{"role": "system", "content": system}] + clean_history + [
        {"role": "user", "content": question}
    ]
    answer = generate_reply(messages)

    if excerpts:
        src = "\n".join(f"- {c[:160]}..." for c in excerpts[:3])
    else:
        src = "- Page 1 overview of the document"
    return f"{answer}\n\n---\n**Sources used:**\n{src}"

def extract_overview(ov):
    """Pull the two key sentences straight out of page 1."""
    s1 = re.search(r"The Government of India has approved.*?\d{4}-\d{2}\.", ov)
    s2 = re.search(r"The Scheme comprises.*?Fund\.", ov)
    return s1.group(0) if s1 else None, s2.group(0) if s2 else None

def rag_chat(message, history):
    question = to_text(message)
    if is_broad(question):
        s1, s2 = extract_overview(overview)
        if s1 and s2:
            return (
                "From page 1 of the document:\n\n"
                f"> {s1}\n\n> {s2}\n\n"
                "---\n**Sources used:**\n- Page 1 of the document (exact text)"
            )
    return llm_chat(message, history)      # specific questions use search + model

gr.ChatInterface(rag_chat, title="GOBARdhan Guidelines Chatbot").launch(share=True, debug=True)